# Check the frozen TEST predictions

This notebook reads the three saved TEST prediction files and checks labels, image indices, probabilities, logits, dataset hashes and checkpoint hashes. It reports accuracy, high-confidence error counts and expected calibration error, or ECE, using 10 confidence bins. High-confidence errors have reported confidence of at least 90%.

The default run does not load models, read training or validation images, download files or overwrite predictions. Run it from the repository root before [02_model_behavior.ipynb](02_model_behavior.ipynb) and [03_robustness.ipynb](03_robustness.ipynb). If a checkpoint is a Git LFS pointer rather than the actual file, run `git lfs pull`. The earlier exports for all dataset splits remain in [archived/analysis_history.ipynb](archived/analysis_history.ipynb).

In [1]:
from pathlib import Path
import hashlib
import numpy as np
from IPython.display import display, Markdown

root=Path.cwd()
with np.load(root/'data/bloodmnist.npz',allow_pickle=False) as z:
    images,labels=z['test_images'],z['test_labels'].ravel()
assert images.shape==(3421,28,28,3) and labels.shape==(3421,)
names=['resnet18','resnet50','vit']
checkpoints=['weights_bloodmnist/resnet18_224_1.pth','weights_bloodmnist/resnet50_28_1.pth','models/vit']
files=checkpoints[:2]+['models/vit/model.safetensors','models/vit/config.json','models/vit/preprocessor_config.json']
def sha256_file(path):
    digest=hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):digest.update(block)
    return digest.hexdigest()
actual_hashes=[sha256_file(root/file) for file in files]
with np.load(root/'predictions/occlusion_test.npz',allow_pickle=False) as z:
    np.testing.assert_array_equal(actual_hashes,z['weight_hashes'])
with np.load(root/'predictions/targeted_followup.npz',allow_pickle=False) as z:
    assert str(z['split'])=='test' and str(z['input_sha256'])==hashlib.sha256(images.tobytes()).hexdigest()
    np.testing.assert_array_equal(z['labels'],labels)
    np.testing.assert_array_equal(actual_hashes,z['weight_hashes'])
    reference=z['baseline_probabilities']
cache=[];rows=['| Model / preprocessing | TEST accuracy | Mean confidence | ECE (10 bins) | HC errors |','|---|---:|---:|---:|---:|']
for m,name in enumerate(names):
    with np.load(root/f'predictions/{name}_test.npz',allow_pickle=False) as z:
        assert str(z['split'])=='test' and str(z['model'])==name and str(z['checkpoint'])==checkpoints[m]
        assert int(z['input_resolution'])==(28 if m==1 else 224)
        np.testing.assert_array_equal(z['labels'],labels)
        np.testing.assert_array_equal(z['sample_index'],np.arange(len(labels)))
        q,logits=z['probabilities'],z['logits']
    assert q.shape==logits.shape==(3421,8) and np.isfinite(q).all() and np.isfinite(logits).all()
    assert ((q>=0)&(q<=1)).all()
    scores=logits.astype(float);scores-=scores.max(-1,keepdims=True);scores=np.exp(scores)
    np.testing.assert_allclose(q,scores/scores.sum(-1,keepdims=True),atol=2e-7,rtol=1e-5)
    np.testing.assert_allclose(q.sum(-1),1,atol=1e-6)
    np.testing.assert_allclose(q,reference[m],atol=1e-8,rtol=1e-6)
    assigned,confidence=q.argmax(-1),q.max(-1);correct=assigned==labels
    groups=np.minimum((confidence*10).astype(int),9)
    ece=sum((groups==b).mean()*abs(correct[groups==b].mean()-confidence[groups==b].mean()) for b in range(10) if (groups==b).any())
    hc=int(((~correct)&(confidence>=.9)).sum());cache.append(q)
    label=['ResNet18 · nearest224','ResNet50 · native28','ViT · local bilinear224'][m]
    rows.append(f'| {label} | {correct.mean():.2%} | {confidence.mean():.2%} | {100*ece:.2f} pp | {hc} |')
display(Markdown('\n'.join(rows)))
print('Verified 3,421 TEST IDs, probability/logit consistency, shared baselines and five checkpoint/configuration hashes.')


| Model / preprocessing | TEST accuracy | Mean confidence | ECE (10 bins) | HC errors |
|---|---:|---:|---:|---:|
| ResNet18 · nearest224 | 96.61% | 98.73% | 2.19 pp | 70 |
| ResNet50 · native28 | 95.73% | 98.66% | 2.98 pp | 79 |
| ViT · local bilinear224 | 96.99% | 98.62% | 1.67 pp | 52 |

Verified 3,421 TEST IDs, probability/logit consistency, shared baselines and five checkpoint/configuration hashes.


ViT has the highest TEST accuracy and fewest high-confidence errors, while ResNet50 has the lowest accuracy and largest ECE. Mean confidence and ECE measure different things, so read them separately. These results compare the supplied checkpoints and preprocessing, not training performance or architecture alone.

## Optional checkpoint replay

Leave `REPLAY_TEST = False` for the saved-results workflow. Set it to `True` to repeat inference on the full TEST set with local PyTorch and Transformers installations and the original preprocessing.

The replay compares probabilities and labels with the saved predictions. It never overwrites them, so changes to a checkpoint or software version cannot silently replace the reported results. Use an isolated Python environment with `requirements.txt` installed.

In [2]:
REPLAY_TEST = False
if REPLAY_TEST:
    import torch
    from torchvision.models import resnet18
    from transformers import ViTForImageClassification, ViTImageProcessor
    from models.resnet18.models import ResNet50
    torch.manual_seed(42);torch.set_num_threads(4);torch.backends.cudnn.benchmark=False
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    models=[resnet18(weights=None,num_classes=8),ResNet50(in_channels=3,num_classes=8)]
    for model,path in zip(models,checkpoints):
        model.load_state_dict(torch.load(root/path,map_location='cpu',weights_only=True)['net'])
    models.append(ViTForImageClassification.from_pretrained(root/'models/vit',local_files_only=True))
    processor=ViTImageProcessor.from_pretrained(root/'models/vit',local_files_only=True)
    for m,model in enumerate(models):
        model=model.to(device).eval();parts=[]
        with torch.inference_mode():
            for start in range(0,len(images),32):
                batch=images[start:start+32]
                if m==2:
                    inputs=processor(images=list(batch),return_tensors='pt').to(device)
                    logits=model(**inputs).logits
                else:
                    x=torch.from_numpy(batch.copy()).permute(0,3,1,2).to(device).float()
                    if m==0:x=torch.nn.functional.interpolate(x,(224,224),mode='nearest')
                    logits=model((x/255-.5)/.5)
                parts.append(logits.softmax(-1).cpu().numpy())
        observed=np.concatenate(parts)
        np.testing.assert_allclose(observed,cache[m],atol=1e-5,rtol=1e-4)
        np.testing.assert_array_equal(observed.argmax(-1),cache[m].argmax(-1))
        print(names[m],': full TEST replay agrees with frozen predictions')
else:
    print('Replay disabled: baseline verification completed from existing TEST caches; no inference or writes.')


Replay disabled: baseline verification completed from existing TEST caches; no inference or writes.
